# 09 · External validation A: controlled temperature test with pythia-2.8b (design doc §8.8.14; protocol A of external_validation_protocol.md)

**How to run**: pick a Colab server with an **A100 GPU** and click "Run All"; about 25 minutes. If the connection drops, click "Run All" again; finished parts are skipped. To rerun in the same runtime, first use Runtime → Disconnect and delete runtime. Results go to `DATA_DIR/09_heldout_pythia/`.

**What it computes**: 1,000 M4GT human texts that the Subtask B manifest does not use (a frozen list with locked hashes). The first 30 tokens of the first 300 are prompts; `EleutherAI/pythia-2.8b` continues them to equal length at nine temperatures (2,700 texts), and Falcon-7B-Instruct with Falcon-7B scores them (as in 07). Primary endpoint: whether the failure temperature predicted by the dual curvature margin lies within 0.02 of the temperature where the Fast-DetectGPT AUROC crosses 0.5.


In [ ]:
# 1. Check that the runtime has an A100: the two Falcon 7B models need about 30 GB.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to an A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(name, f"{mem:.0f} GB")
assert mem > 35, f"This {name} has only {mem:.0f} GB, not enough for two 7B models. Choose an A100 GPU."
free = torch.cuda.mem_get_info()[0] / 2**30
assert free > 35, f"Only {free:.0f} GB of GPU memory is free: another notebook is using this GPU. Go to Runtime → Manage sessions, terminate the other sessions, and rerun."
print(f"{free:.0f} GB free")


In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
OUT = "09_heldout_pythia"
os.makedirs(OUT, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"


In [ ]:
# 3. Install packages (as in 06 and 07).
%pip install -q -U transformers huggingface-hub accelerate
import json, gc, glob, shutil, time, hashlib, sys, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd
import transformers


In [ ]:
# 4. Check the hashes of the raw file and the human manifest (stop on a mismatch), then load the 1,000 human texts and check each text hash.
RAW = "external/m4gt/SubtaskB.jsonl"; MAN = "manifests/m4gt_humans_A_manifest.csv"
EXPECTED_RAW = "4d65fc1fb93b5c9d21b226aca06b610ee031fa9ccbc18fa6f39d16273dd281c7"
EXPECTED_MAN = "7c6f2b6529a8ef7b1a1920bc798d45c68a7a7162823566f52a73deb6ae8d1caf"

def sha256_file(path):
    d = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            d.update(block)
    return d.hexdigest()

assert sha256_file(RAW) == EXPECTED_RAW, "Raw file SHA-256 does not match; stopping."
assert sha256_file(MAN) == EXPECTED_MAN, "Human manifest SHA-256 does not match; stopping."
man = pd.read_csv(MAN)
assert len(man) == 1000 and man.id.tolist() == list(range(1000)) and (man.role[:300] == "source").all()
need = {int(r.line_number): r for r in man.itertuples(index=False)}
found = {}
with open(RAW, encoding="utf-8") as f:
    for ln, line in enumerate(f, start=1):
        r = need.get(ln)
        if r is None:
            continue
        it = json.loads(line)
        assert hashlib.sha256(it["text"].encode("utf-8")).hexdigest() == r.text_sha256, f"Text hash mismatch on line {ln}"
        assert it["source"] == r.source and it["model"] == "human"
        found[int(r.id)] = dict(id=int(r.id), domain=r.source, text=it["text"])
assert len(found) == 1000
humans = [found[i] for i in range(1000)]; sources = humans[:300]
print(len(humans), "human texts; prompts:", len(sources), pd.Series([h["domain"] for h in sources]).value_counts().to_dict())


In [ ]:
# 5. Batched continuations: pythia-2.8b at nine temperatures. Same rules as 06: first 30 tokens, plain softmax sampling, temperature, top-p 1.0, top-k off,
#    no repetition penalty, special and end-of-text tokens banned, length equal to the human continuation in tokens. Each temperature is saved when done.
from transformers import AutoTokenizer, AutoModelForCausalLM
GEN = "EleutherAI/pythia-2.8b"; TAG = GEN.split("/")[1]
TEMPS = [0.8, 0.9, 0.94, 0.96, 0.98, 1.0, 1.02, 1.04, 1.1]
PREFIX, MAX_TOK, BS, SEED = 30, 512, 24, 42

def prepare(tok, texts):
    items = []
    for h in texts:
        ids = tok(h["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) < PREFIX + 20:
            continue
        items.append(dict(id=h["id"], domain=h["domain"], prefix_ids=ids[:PREFIX], target=len(ids) - PREFIX,
                          human=tok.decode(ids[PREFIX:], clean_up_tokenization_spaces=False),
                          prefix=tok.decode(ids[:PREFIX], clean_up_tokenization_spaces=False)))
    return items

@torch.no_grad()
def generate_batch(model, tok, batch, temp, special):
    pad = tok.pad_token_id
    L = max(len(b["prefix_ids"]) for b in batch); new = max(b["target"] for b in batch)
    x = torch.full((len(batch), L), pad); att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        x[i, L - len(b["prefix_ids"]):] = torch.tensor(b["prefix_ids"]); att[i, L - len(b["prefix_ids"]):] = 1
    out = model.generate(input_ids=x.cuda(), attention_mask=att.cuda(), do_sample=True, temperature=temp, top_p=1.0, top_k=0,
                         max_new_tokens=new, min_new_tokens=new, suppress_tokens=special, pad_token_id=pad,
                         repetition_penalty=1.0)
    gen = out[:, L:].cpu()
    res = [gen[i, :b["target"]].tolist() for i, b in enumerate(batch)]
    for b, ids in zip(batch, res):   # self-check: exact length, no special tokens
        assert len(ids) == b["target"], ("wrong length", len(ids), b["target"])
        assert not (set(ids) & set(special)), "a special token was generated"
    return res

todo = [t for t in TEMPS if not os.path.exists(f"{OUT}/gen_{TAG}_T{t}.json")]
hpath = f"{OUT}/humans_{TAG}.json"
if todo or not os.path.exists(hpath):
    globals().pop("model", None); gc.collect(); torch.cuda.empty_cache()   # on a rerun of this cell, free the pythia loaded last time
    tok = AutoTokenizer.from_pretrained(GEN); tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(GEN, dtype=torch.bfloat16).cuda().eval()
    special = sorted(set(tok.all_special_ids) | {tok.eos_token_id} | set(np.atleast_1d(model.generation_config.eos_token_id).tolist() if model.generation_config.eos_token_id is not None else []))
    items = sorted(prepare(tok, sources), key=lambda b: b["target"])
    assert len(items) == 300, f"Only {len(items)} prompts left; some texts are too short"
    for temp in todo:
        torch.manual_seed(SEED); t0 = time.time(); rows = []
        for s in range(0, len(items), BS):
            batch = items[s:s + BS]
            for b, ids in zip(batch, generate_batch(model, tok, batch, temp, special)):
                rows.append(dict(id=b["id"], domain=b["domain"], model=GEN, decoding=f"T{temp}", prefix=b["prefix"],
                                 ai=tok.decode(ids, clean_up_tokenization_spaces=False), human=b["human"], n_tokens=len(ids)))
        json.dump(rows, open(f"{OUT}/gen_{TAG}_T{temp}.json", "w"), ensure_ascii=False)
        print(f"{TAG} T={temp}: {len(rows)} texts, {(time.time() - t0) / 60:.1f} min", flush=True)
    if not os.path.exists(hpath):
        hum = prepare(tok, humans); assert len(hum) == 1000, f"Only {len(hum)} human texts left"
        json.dump([dict(id=b["id"], domain=b["domain"], prefix=b["prefix"], human=b["human"]) for b in hum], open(hpath, "w"), ensure_ascii=False)
    del model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/hf/models--{GEN.replace('/', '--')}", ignore_errors=True)
else:
    print("Generation already done, skipping")


In [ ]:
# 6. Falcon dual-model scoring (as in 07_falcon_dual_scoring.ipynb): scorer Falcon-7B-Instruct, observer Falcon-7B, 12 rows per position, continuation only.
SCORER, OBSERVER, SBS = "tiiuae/falcon-7b-instruct", "tiiuae/falcon-7b", 8
spath = f"{OUT}/scores_falcon_{TAG}.pkl"

def records_for():
    recs = []
    for f in sorted(glob.glob(f"{OUT}/gen_{TAG}_T*.json")):
        for r in json.load(open(f)):
            recs.append(dict(id=r["id"], domain=r["domain"], model=GEN, decoding=r["decoding"], who="ai", prefix=r["prefix"], text=r["ai"]))
    for r in json.load(open(hpath)):
        recs.append(dict(id=r["id"], domain=r["domain"], model=GEN, decoding="human", who="human", prefix=r["prefix"], text=r["human"]))
    return recs

@torch.no_grad()
def rank_and_cdf(lp, tgt):
    obs = lp.gather(1, tgt[:, None])
    rank = (lp > obs).sum(-1).float()
    p = lp.exp()
    cdf = (p * (lp < obs)).sum(-1)
    return rank, cdf

@torch.no_grad()
def score_records(recs, out_path):
    if os.path.exists(out_path):
        print("Already done, skipping:", out_path); return
    for _name in ("p_model", "q_model"):   # on a rerun of this cell, free the models loaded last time
        globals().pop(_name, None)
    gc.collect(); torch.cuda.empty_cache()
    tok = AutoTokenizer.from_pretrained(SCORER); tok.padding_side = "right"
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    p_model = AutoModelForCausalLM.from_pretrained(SCORER, dtype=torch.bfloat16).cuda().eval()
    q_model = AutoModelForCausalLM.from_pretrained(OBSERVER, dtype=torch.bfloat16).cuda().eval()
    print("loaded", f"{torch.cuda.memory_allocated() / 2**30:.1f} GB")
    enc = []
    for r in recs:
        k = len(tok(r["prefix"])["input_ids"])
        ids = tok(r["prefix"] + r["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) - k >= 10:
            enc.append((r, k, ids))
    enc.sort(key=lambda e: len(e[2]))
    out, t0 = [], time.time()
    for s in range(0, len(enc), SBS):
        chunk = enc[s:s + SBS]; L = max(len(e[2]) for e in chunk)
        x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, (_, _, ids) in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
        x, att = x.cuda(), att.cuda()
        lg_p = p_model(input_ids=x, attention_mask=att).logits
        lg_q = q_model(input_ids=x, attention_mask=att).logits
        for i, (r, k, ids) in enumerate(chunk):
            n = len(ids)
            lp = torch.log_softmax(lg_p[i, :n - 1].float(), -1); lq = torch.log_softmax(lg_q[i, :n - 1].float(), -1)
            tgt = torch.tensor(ids[1:], device=lp.device)
            p = lp.exp(); q = lq.exp()
            e_p = (p * lp).sum(-1); X = (q * lp).sum(-1)
            rank_p, cdf_p = rank_and_cdf(lp, tgt); rank_q, cdf_q = rank_and_cdf(lq, tgt)
            rows = [-lp.gather(1, tgt[:, None])[:, 0], -e_p, (p * lp * lp).sum(-1) - e_p ** 2, p.topk(10, -1).values.sum(-1),
                    X, (q * lp * lp).sum(-1) - X ** 2, -lq.gather(1, tgt[:, None])[:, 0],
                    rank_p, cdf_p, rank_q, cdf_q, -(q * lq).sum(-1)]
            arr = torch.stack(rows).cpu().numpy().astype(np.float32)[:, k - 1:]
            out.append(dict(id=r["id"], domain=r["domain"], model=r["model"], decoding=r["decoding"], who=r["who"], arr=arr))
        if (s // SBS) % 50 == 0:
            print(f"  {s}/{len(enc)} {(time.time() - t0) / 60:.1f} min", flush=True)
    pd.DataFrame(out).to_pickle(out_path)
    print("saved", out_path, len(out), f"{(time.time() - t0) / 60:.1f} min")
    del p_model, q_model; gc.collect(); torch.cuda.empty_cache()

score_records(records_for(), spath)


In [ ]:
# 7. Quick summary and primary endpoint. The full analysis (CIs, per source) is scripts/e5_heldout_pythia.py.
from sklearn.metrics import roc_auc_score

def cross(x, y, level=0.0):
    x, y = np.asarray(x, float), np.asarray(y, float) - level
    for i in range(len(x) - 1):
        if y[i] > 0 >= y[i + 1] or y[i] < 0 <= y[i + 1]:
            return float(x[i] + (x[i + 1] - x[i]) * y[i] / (y[i] - y[i + 1]))
    return None

s = pd.read_pickle(spath)
s["fdg"] = [(-a[0].sum() - a[4].sum()) / np.sqrt(a[5].sum()) for a in s.arr]
s["bino"] = [-(a[0].mean() / -a[4].mean()) for a in s.arr]
s["surprisal"] = [a[0].mean() for a in s.arr]
s["curv_dual"] = [(-a[4] - a[0]).mean() for a in s.arr]     # observer cross-entropy − surprisal (the Fast-DetectGPT numerator)
s["curv_self"] = [(a[1] - a[0]).mean() for a in s.arr]      # the scorer's own entropy − surprisal
h = s[s.who == "human"]; rows = []
for dec, g in s[s.who == "ai"].groupby("decoding"):
    y = np.r_[np.ones(len(g)), np.zeros(len(h))]
    rows.append(dict(temperature=float(dec[1:]), n_ai=len(g), n_human=len(h),
                     AUROC_FDG=roc_auc_score(y, np.r_[g.fdg.values, h.fdg.values]), AUROC_Bino=roc_auc_score(y, np.r_[g.bino.values, h.bino.values]),
                     curvature_margin_dual=g.curv_dual.mean() - h.curv_dual.mean(), curvature_margin_self=g.curv_self.mean() - h.curv_self.mean(),
                     surprisal_margin=h.surprisal.mean() - g.surprisal.mean()))
R = pd.DataFrame(rows).sort_values("temperature"); R.to_csv(f"{OUT}/summary.csv", index=False)
print(R.round(3).to_string(index=False))
ac, ab = cross(R.temperature, R.AUROC_FDG, 0.5), cross(R.temperature, R.AUROC_Bino, 0.5)
pc, ps, pm = cross(R.temperature, R.curvature_margin_dual), cross(R.temperature, R.curvature_margin_self), cross(R.temperature, R.surprisal_margin)
fmt = lambda v: "outside grid" if v is None else f"{v:.3f}"
print(f"\nObserved failure temperature: Fast-DetectGPT {fmt(ac)}, Binoculars {fmt(ab)}; predicted: dual curvature margin {fmt(pc)}, own-entropy curvature margin {fmt(ps)}, surprisal margin {fmt(pm)}")
if ac is None or pc is None:
    print("Primary endpoint: the crossing is outside the 0.8 to 1.1 grid, so it cannot be judged (no temperatures may be added afterwards)")
else:
    err = abs(ac - pc)
    print(f"Primary endpoint: error {err:.3f} -> {'met' if err <= 0.02 else 'not met'} (pre-registered threshold 0.02)")
if ac is not None and ab is not None:
    print(f"Secondary endpoint: the two crossings differ by {abs(ac - ab):.3f}")
run_info = dict(completed_at_utc=datetime.now(timezone.utc).isoformat(), raw_sha256=EXPECTED_RAW, humans_manifest_sha256=EXPECTED_MAN,
                generator=GEN, temperatures=TEMPS, prefix_tokens=PREFIX, gen_seed=SEED, scorer=SCORER, observer=OBSERVER, max_tokens=MAX_TOK,
                python=sys.version, platform=platform.platform(), torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(run_info, open(f"{OUT}/run_info.json", "w"), ensure_ascii=False, indent=2)
print("outputs:", sorted(os.listdir(OUT)))


In [ ]:
# 8. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/09_heldout_pythia/")
